# Game AI & Procedural Generation

Fully self-contained: numpy + matplotlib only (noise is implemented from scratch so
no extra packages needed). Each algorithm here plugs directly into the pygame subtopic.

## 1. A* on a 2D grid with obstacles

In [ ]:
import heapq

import numpy as np

def astar(grid, start, goal):
    """grid: 2D array, 0=floor 1=wall. Returns list of (r, c) or None."""
    rows, cols = grid.shape

    def heuristic(a, b):                       # Manhattan — admissible on 4-connected grids
        return abs(a[0] - b[0]) + abs(a[1] - b[1])

    frontier = [(heuristic(start, goal), 0, start)]   # (f, g, node)
    came_from = {start: None}
    g_score = {start: 0}

    while frontier:
        _, g, current = heapq.heappop(frontier)
        if current == goal:
            path = []
            while current is not None:
                path.append(current)
                current = came_from[current]
            return path[::-1]
        for dr, dc in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
            nxt = (current[0] + dr, current[1] + dc)
            if not (0 <= nxt[0] < rows and 0 <= nxt[1] < cols) or grid[nxt] == 1:
                continue
            new_g = g + 1
            if new_g < g_score.get(nxt, float("inf")):
                g_score[nxt] = new_g
                came_from[nxt] = current
                heapq.heappush(frontier, (new_g + heuristic(nxt, goal), new_g, nxt))
    return None

grid = np.zeros((20, 30), dtype=int)
grid[5:15, 10] = 1     # a wall with a gap
grid[5, 10:20] = 1
path = astar(grid, (10, 2), (10, 27))
print(f"path found, length {len(path)}")

## 2. Visualize the path

In [ ]:
import matplotlib.pyplot as plt

def show_path(grid, path, title="A* path"):
    img = grid.astype(float).copy()
    for r, c in path or []:
        img[r, c] = 0.5
    img[path[0]] = 0.25; img[path[-1]] = 0.75
    plt.figure(figsize=(9, 5))
    plt.imshow(img, cmap="magma")
    plt.title(title); plt.axis("off"); plt.show()

show_path(grid, path)

## 3. Value noise from scratch (Perlin's smooth-randomness idea)

In [ ]:
def value_noise(shape, res, seed=0):
    """Random values on a coarse lattice, smoothly interpolated up to `shape`."""
    rng = np.random.default_rng(seed)
    lattice = rng.random((res + 1, res + 1))
    ys = np.linspace(0, res, shape[0], endpoint=False)
    xs = np.linspace(0, res, shape[1], endpoint=False)
    y0, x0 = ys.astype(int), xs.astype(int)
    ty, tx = ys - y0, xs - x0
    ty = ty * ty * (3 - 2 * ty)          # smoothstep fade (this is Perlin's trick)
    tx = tx * tx * (3 - 2 * tx)
    ty, tx = ty[:, None], tx[None, :]
    a = lattice[np.ix_(y0, x0)]
    b = lattice[np.ix_(y0, x0 + 1)]
    c = lattice[np.ix_(y0 + 1, x0)]
    d = lattice[np.ix_(y0 + 1, x0 + 1)]
    return (a * (1 - tx) + b * tx) * (1 - ty) + (c * (1 - tx) + d * tx) * ty

def fractal_noise(shape, octaves=4, seed=0):
    total, amplitude, res, norm = np.zeros(shape), 1.0, 4, 0.0
    for o in range(octaves):
        total += amplitude * value_noise(shape, res, seed + o)
        norm += amplitude
        amplitude *= 0.5          # each octave: half the strength...
        res *= 2                  # ...twice the detail
    return total / norm

height = fractal_noise((128, 128), octaves=5, seed=3)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
axes[0].imshow(height, cmap="terrain"); axes[0].set_title("height map (5 octaves)")
biomes = np.digitize(height, [0.35, 0.45, 0.6, 0.75])  # water/sand/grass/rock/snow
axes[1].imshow(biomes, cmap="terrain"); axes[1].set_title("thresholded into biomes")
[ax.axis("off") for ax in axes]; plt.show()

## 4. Cellular automata: cave generation

In [ ]:
def generate_cave(shape=(50, 80), fill=0.45, passes=4, seed=1):
    rng = np.random.default_rng(seed)
    cave = (rng.random(shape) < fill).astype(int)
    cave[0, :] = cave[-1, :] = cave[:, 0] = cave[:, -1] = 1   # solid border

    for _ in range(passes):
        # count the 8 neighbors of every cell at once (vectorized)
        padded = np.pad(cave, 1, constant_values=1)
        neighbors = sum(padded[1 + dr:shape[0] + 1 + dr, 1 + dc:shape[1] + 1 + dc]
                        for dr in (-1, 0, 1) for dc in (-1, 0, 1)
                        if (dr, dc) != (0, 0))
        cave = (neighbors >= 5).astype(int)                    # the one local rule
        cave[0, :] = cave[-1, :] = cave[:, 0] = cave[:, -1] = 1
    return cave

fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
for ax, passes in zip(axes, [0, 2, 4]):
    ax.imshow(generate_cave(passes=passes), cmap="gray_r")
    ax.set_title(f"{passes} smoothing passes"); ax.axis("off")
plt.show()

## 5. NPC finite state machine: patrol → chase → return

In [ ]:
class GuardFSM:
    CHASE_RANGE, LOSE_RANGE = 6.0, 10.0

    def __init__(self, waypoints):
        self.state = "PATROL"
        self.waypoints, self.wp_index = waypoints, 0
        self.pos = np.array(waypoints[0], dtype=float)

    def update(self, player_pos):
        distance = np.linalg.norm(self.pos - player_pos)
        # transitions — explicit and testable
        if self.state == "PATROL" and distance < self.CHASE_RANGE:
            self.state = "CHASE"
        elif self.state == "CHASE" and distance > self.LOSE_RANGE:
            self.state = "PATROL"
        # per-state behavior
        target = (np.array(player_pos, dtype=float) if self.state == "CHASE"
                  else np.array(self.waypoints[self.wp_index], dtype=float))
        step = target - self.pos
        if np.linalg.norm(step) < 0.5:
            if self.state == "PATROL":
                self.wp_index = (self.wp_index + 1) % len(self.waypoints)
        else:
            self.pos += step / np.linalg.norm(step) * 0.8
        return self.state

guard = GuardFSM(waypoints=[(0, 0), (10, 0), (10, 10), (0, 10)])
player_path = [(20, 20)] * 10 + [(9, 4)] * 12 + [(25, 25)] * 12   # approach, then flee

log = [guard.update(np.array(p, dtype=float)) for p in player_path]
transitions = [f"t={i}: {a}->{b}" for i, (a, b) in enumerate(zip(log, log[1:])) if a != b]
print(" | ".join(transitions))

## 6. Mini-project: procedural level + A* across it

In [ ]:
cave = generate_cave(shape=(50, 80), seed=7)
floor_cells = np.argwhere(cave == 0)

# pick far-apart start/goal on open floor
start = tuple(floor_cells[np.argmin(floor_cells.sum(axis=1))])
goal = tuple(floor_cells[np.argmax(floor_cells.sum(axis=1))])
cave_path = astar(cave, start, goal)

if cave_path:
    show_path(cave, cave_path, title=f"A* through a generated cave ({len(cave_path)} steps)")
else:
    print("start and goal are in disconnected caverns — the flood-fill exercise below fixes this")

# Extension exercises:
# 1. Flood-fill to find connected caverns; keep only the largest (or tunnel between them).
# 2. Use diagonal movement (8-connected) with octile-distance heuristic.
# 3. Render this live in pygame: guard uses GuardFSM + astar to hunt the player.